# Serving an embedding model with vLLM

```
clients ──ngrok──▶ FastAPI proxy ──▶ vLLM (localhost only)
                   │ per-client API key
                   └ per-key rate limiting
```

vLLM is never exposed directly: it has no authentication of its own, so the
proxy is the only entry point and vLLM listens on localhost.

One key per client. A runaway loop exhausts that client's rate budget rather
than everyone's, and the log shows whose traffic it is.

## 1. Dependencies

In [1]:
%%capture
!pip install -qq fastapi==0.135.2 uvicorn==0.42.0 pyngrok==7.5.1
!pip install -qq vllm==0.27.1
!pip uninstall -qq -y torchaudio

## 2. Imports

In [2]:
from __future__ import annotations

import json
import os
import signal
import socket
import subprocess
import sys
import time
from dataclasses import dataclass
from threading import Thread
from typing import List, Optional, Sequence

import torch
from pyngrok import ngrok
from huggingface_hub import login, snapshot_download

## 3. Secrets

Two credentials: a Hugging Face token to fetch the weights, and an ngrok token
to open the tunnel.

The Colab Secrets panel is a feature of the Colab web page, not of the runtime,
so `userdata` does not exist when the same runtime is driven from an IDE plugin.
Pasting them works in both.

In [3]:
# Two Colab Secrets, set once from the key icon in the left sidebar and enabled
# for this notebook. They are read rather than pasted on purpose: a token typed
# into a cell is saved with the notebook, and this one is in a repository.
#
#   HF_TOKEN     https://huggingface.co/settings/tokens
#                fine-grained, read-only, limited to the model repo below
#   NGROK_TOKEN  https://dashboard.ngrok.com -> Authtokens
from google.colab import userdata
HF_TOKEN, NGROK_TOKEN = userdata.get("HF_TOKEN"), userdata.get("NGROK_TOKEN")

assert HF_TOKEN and NGROK_TOKEN, (
    "set HF_TOKEN and NGROK_TOKEN as Colab Secrets (key icon, left sidebar) "
    "and switch both on for this notebook")

os.environ["HF_TOKEN"] = HF_TOKEN
login(HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)
print("hugging face and ngrok authenticated")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


hugging face and ngrok authenticated                                                                


## 4. Configuration

In [4]:
MODEL = 'Qwen/Qwen3-Embedding-0.6B' # @param {type:'string'}
# Pinned to a commit, not to a branch: a Hugging Face repo can be updated under
# the same name, and the vector a model produces is not stable across versions.
# An index built from one revision cannot be compared with one built from
# another, and nothing in the numbers would say so.
REVISION = '97b0c614be4d' # @param {type:'string'}

# vLLM detects a pooling model on its own, so this is usually unnecessary. Set
# anyway: a notebook that states what it is doing survives the version where
# the detection changes.
RUNNER = 'pooling' # @param {type:'string'}

# A reserved domain, so that a restart does not move the address clients use.
NGROK_DOMAIN = os.environ.get("NGROK_DOMAIN") or 'kgtp-embed.ngrok-free.app' # @param {type:'string'}

VLLM_PORT = 8000      # @param {type:'integer'}
PROXY_PORT = 8899     # @param {type:'integer'}
MAX_MODEL_LEN = 4096  # @param {type:'integer'}
GPU_MEM_UTIL = 0.85   # @param {type:'number'}

# One key per client. The name shows up in the proxy log.
API_KEYS = {
    "student-key-01": "alice",
    "student-key-02": "bob",
    "teacher-key": "giuseppe",
}
# Indexing a corpus is thousands of short calls, not a conversation, so the
# limit that protects a generative endpoint would stop the job here.
RATE_LIMIT_RPM = 600  # @param {type:'integer'}

print(f"model:  {MODEL}@{REVISION}")
print(f"domain: {NGROK_DOMAIN}")

model:  Qwen/Qwen3-Embedding-0.6B@97b0c614be4d
domain: kgtp-embed.ngrok-free.app


## 5. The GPU, and the weights

The download is a step of its own so that its progress is visible and
resumable. vLLM opens its port only once the model is loaded, so a client
waiting on that port cannot tell a slow download from a hung server, and a
timeout there kills the download halfway.

In [5]:
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Change the Colab runtime type to a GPU.")

name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name}, {total:.0f} GB")

path = snapshot_download(repo_id=MODEL, revision=REVISION, token=HF_TOKEN, max_workers=8)
print("weights at", path)

GPU: NVIDIA A100-SXM4-80GB, 79 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

weights at /root/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3


## 6. vLLM

In [6]:
@dataclass
class VLLMConfig:
    model: str = MODEL
    revision: str = REVISION
    runner: str = RUNNER
    host: str = "127.0.0.1"   # never exposed: the proxy sits in front
    port: int = VLLM_PORT
    max_model_len: int = MAX_MODEL_LEN
    gpu_memory_utilization: float = GPU_MEM_UTIL

    @property
    def base_url(self) -> str:
        return f"http://{self.host}:{self.port}"


def select_dtype() -> str:
    major, _ = torch.cuda.get_device_capability(0)
    # Cards below compute capability 8 have no bfloat16 support, and asking for
    # it fails at load with a message about the dtype rather than about the card.
    return "bfloat16" if major >= 8 else "float16"


def build_args(cfg: VLLMConfig) -> Sequence[str]:
    return [
        "vllm", "serve", cfg.model,
        "--revision", cfg.revision,
        "--runner", cfg.runner,
        "--dtype", select_dtype(),
        "--max-model-len", str(cfg.max_model_len),
        "--gpu-memory-utilization", str(cfg.gpu_memory_utilization),
        "--host", cfg.host,
        "--port", str(cfg.port),
    ]


def _pump(pipe, tag: str):
    try:
        for line in iter(pipe.readline, ""):
            if not line:
                break
            print(f"[{tag}] {line}", end="")
    finally:
        try:
            pipe.close()
        except Exception:
            pass


def start_process(argv, tag, cwd=None) -> subprocess.Popen:
    proc = subprocess.Popen(
        argv, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
        start_new_session=True, bufsize=1, env=os.environ.copy(),
    )
    Thread(target=_pump, args=(proc.stdout, tag), daemon=True).start()
    return proc


def stop_process(proc: Optional[subprocess.Popen]) -> None:
    """Stop a process and everything it forked.

    vLLM runs its engine in child processes and those children hold the weights.
    Signalling only the parent leaves them alive with the VRAM still allocated,
    and the next launch fails on an out-of-memory error that looks like a
    configuration problem and is not. start_new_session put the whole tree in
    one process group, which is what makes it addressable here.
    """
    if not proc:
        return
    try:
        pgid = os.getpgid(proc.pid)
    except (ProcessLookupError, PermissionError):
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(pgid, signal.SIGKILL)
            proc.wait(timeout=10)
    except Exception as e:
        print(f"error while stopping {proc.pid}: {e}")


def wait_for_port(host, port, timeout=900.0, proc=None) -> bool:
    """Wait for the port, and give up early if the process already exited.

    Without the process check a crash on startup is indistinguishable from a
    slow load, and the caller blocks for the full timeout before reporting an
    error that says nothing.
    """
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            print(f"process exited with code {proc.returncode} before opening the port")
            return False
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1.0)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(0.5)
    return False

## 7. The proxy

Auth and rate limiting, and nothing else. An embedding call returns a
fixed-size vector in milliseconds, so there is no streaming to relay and no
concurrency cap to enforce: the queue that protects a generative endpoint from
one long generation would protect nothing here.

In [7]:
with open("/content/proxy_config.json", "w") as f:
    json.dump({"vllm_url": f"http://127.0.0.1:{VLLM_PORT}",
               "api_keys": API_KEYS,
               "rate_limit_rpm": RATE_LIMIT_RPM}, f)
print("proxy_config.json written")

proxy_config.json written


In [8]:
%%writefile /content/embed_proxy.py
"""Educational reverse proxy for a vLLM embedding endpoint: auth, rate limiting."""
import json
import time
from collections import defaultdict, deque

import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse

CFG = json.load(open("/content/proxy_config.json"))
API_KEYS = CFG["api_keys"]
RATE_LIMIT_RPM = CFG["rate_limit_rpm"]

app = FastAPI(title="Educational vLLM embedding proxy")
request_log = defaultdict(deque)
client = httpx.AsyncClient(base_url=CFG["vllm_url"], timeout=120)


def check_auth(request: Request) -> str:
    auth = request.headers.get("authorization", "")
    if not auth.startswith("Bearer "):
        raise HTTPException(401, "Missing Authorization: Bearer <key> header")
    key = auth.removeprefix("Bearer ").strip()
    if key not in API_KEYS:
        raise HTTPException(403, "Invalid API key")
    return key


def check_rate_limit(key: str) -> None:
    now = time.monotonic()
    window = request_log[key]
    while window and now - window[0] > 60:
        window.popleft()
    if len(window) >= RATE_LIMIT_RPM:
        retry = int(60 - (now - window[0])) + 1
        raise HTTPException(429, f"Limit of {RATE_LIMIT_RPM} req/min exceeded. Retry in {retry}s.")
    window.append(now)


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post("/v1/embeddings")
async def embeddings(request: Request):
    key = check_auth(request)
    check_rate_limit(key)
    body = await request.body()
    try:
        resp = await client.post("/v1/embeddings", content=body,
                                 headers={"content-type": "application/json"})
        payload = resp.json()
    except Exception as e:
        # A non-JSON body from vLLM is an error page, not an embedding. Saying so
        # here keeps it from surfacing as a parse error in the student's client.
        raise HTTPException(502, f"Error reaching vLLM: {e}")
    n = len(payload.get("data", []) or [])
    print(f"[{time.strftime('%H:%M:%S')}] {API_KEYS[key]} -> {n} embedding(s)", flush=True)
    return JSONResponse(payload, status_code=resp.status_code)


@app.get("/v1/models")
async def models(request: Request):
    check_auth(request)
    resp = await client.get("/v1/models")
    return JSONResponse(resp.json(), status_code=resp.status_code)

Writing /content/embed_proxy.py


## 8. Public serving

The tunnel goes up only after `/health` has answered and the process we started
is still alive. Opening it earlier can publish something left over from an
interrupted run, which from the outside is indistinguishable from success.

In [9]:
import urllib.request

cfg = VLLMConfig()
vllm_proc = start_process(build_args(cfg), "vLLM")
proxy_proc = None
try:
    if not wait_for_port(cfg.host, cfg.port, proc=vllm_proc):
        raise RuntimeError("vLLM did not open the port -- see the log above for why")
    print(f"vLLM up at {cfg.base_url}")

    proxy_proc = start_process(
        ["uvicorn", "embed_proxy:app", "--host", "127.0.0.1", "--port", str(PROXY_PORT)],
        "PROXY", cwd="/content")

    healthy, deadline = False, time.time() + 60
    while time.time() < deadline:
        if proxy_proc.poll() is not None:
            raise RuntimeError("the proxy died on startup -- see the log above")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PROXY_PORT}/health", timeout=5) as r:
                healthy = json.loads(r.read()).get("status") == "ok"
        except Exception:
            healthy = False
        if healthy:
            break
        time.sleep(1)
    if not healthy:
        raise RuntimeError("the proxy did not become healthy within 60s")
    print(f"proxy healthy on :{PROXY_PORT}")

    public = ngrok.connect(PROXY_PORT, "http", domain=NGROK_DOMAIN).public_url
    print("Endpoint for students:", public.rstrip("/") + "/v1")
    print("Health check:", public.rstrip("/") + "/health")

    while True:
        time.sleep(1)
finally:
    try:
        ngrok.kill()
    except Exception:
        pass
    stop_process(proxy_proc)
    stop_process(vllm_proc)

[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345] 
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345]        █     █     █▄   ▄█
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.27.1
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen3-Embedding-0.6B
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:345] 
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:26 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3-Embedding-0.6B', 'host': '127.0.0.1', 'model': 'Qwen/Qwen3-Embedding-0.6B', 'runner': 'pooling', 'dtype': 'bfloat16', 'revision': '97b0c614be4d', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85}
[vLLM] (APIServer pid=2659) INFO 09-01 06:57:42 [model.py:1111] Resolved `--convert auto` to `--convert embed`. Pass the valu

KeyboardInterrupt: 

## 9. Example client

```python
import requests

r = requests.post(
    "https://kgtp-embed.ngrok-free.app/v1/embeddings",
    headers={"Authorization": "Bearer student-key-01",
             "ngrok-skip-browser-warning": "1"},
    json={"model": "Qwen/Qwen3-Embedding-0.6B",
          "input": ["Which contracts did Beat the Streets Chicago receive?"]},
)
vector = r.json()["data"][0]["embedding"]
len(vector)   # 1024
```

The same call embeds a corpus, one batch of `input` at a time.

`/health` answers without a key, which is enough to tell "the endpoint is down"
from "my key is wrong".